# Moirai: Any-variate подход от Salesforce

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/34_moirai.ipynb)

## Установка зависимостей

In [ ]:
!pip install -q uni2ts gluonts torch pandas numpy matplotlib

## Подготовка данных

In [ ]:
import torch
import pandas as pd
import numpy as np

# Создаём многомерные данные
np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')

# Несколько связанных рядов (температура, влажность, давление)
base = np.cumsum(np.random.randn(365))

temperature = 20 + base * 0.5 + 10 * np.sin(np.arange(365) / 365 * 2 * np.pi) + np.random.randn(365) * 2
humidity = 60 - base * 0.3 + 15 * np.cos(np.arange(365) / 365 * 2 * np.pi) + np.random.randn(365) * 5
pressure = 1013 + base * 0.1 + np.random.randn(365) * 3

# Формируем DataFrame в long format
series_data = []
for name, values in [('temperature', temperature), ('humidity', humidity), ('pressure', pressure)]:
    series_data.append(pd.DataFrame({
        'unique_id': name,
        'ds': dates,
        'y': values
    }))

df = pd.concat(series_data, ignore_index=True)
print(f"Всего рядов: {df['unique_id'].nunique()}")
print(df.head(10))

## Moirai: загрузка и прогнозирование

In [ ]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

# Параметры
prediction_length = 30
context_length = 512

# Создаём GluonTS датасет
dataset = PandasDataset.from_long_dataframe(
    df, 
    target='y',
    item_id='unique_id'
)

# Разделяем на train/test
train, test_template = split(dataset, offset=-prediction_length)

print(f"Prediction length: {prediction_length}")
print(f"Context length: {context_length}")

In [ ]:
# Загружаем модель Moirai
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained("Salesforce/moirai-1.0-R-large"),
    prediction_length=prediction_length,
    context_length=context_length,
    patch_size='auto',  # автоматический выбор размера патча
    num_samples=100,    # количество сэмплов для вероятностного прогноза
)

print("Модель загружена!")

In [ ]:
# Генерируем прогнозы
predictor = model.create_predictor(batch_size=32)
forecasts = list(predictor.predict(test_template.input))

print(f"Количество прогнозов: {len(forecasts)}")
print(f"Форма первого прогноза: {forecasts[0].samples.shape}")

## Визуализация прогнозов

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

series_names = ['temperature', 'humidity', 'pressure']

for i, (ax, name) in enumerate(zip(axes, series_names)):
    # История
    history = df[df['unique_id'] == name].tail(100)
    ax.plot(history['ds'], history['y'], 'b-', label='История')
    
    # Прогноз
    forecast = forecasts[i]
    forecast_dates = pd.date_range(
        start=history['ds'].iloc[-1] + pd.Timedelta(days=1),
        periods=prediction_length,
        freq='D'
    )
    
    # Медиана и интервалы
    median = forecast.median
    p10 = forecast.quantile(0.1)
    p90 = forecast.quantile(0.9)
    
    ax.plot(forecast_dates, median, 'r-', label='Медиана')
    ax.fill_between(forecast_dates, p10, p90, 
                    alpha=0.3, color='red', label='80% интервал')
    
    ax.axvline(x=history['ds'].iloc[-1], color='gray', linestyle='--', alpha=0.5)
    ax.set_title(f'{name.capitalize()}')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle('Moirai: прогнозы для многомерных данных', fontsize=14)
plt.tight_layout()
plt.show()

## Размеры патчей для разных частот

In [ ]:
# Таблица соответствий частот и размеров патчей в Moirai
patch_size_table = pd.DataFrame({
    'Частота данных': ['Секунды, минуты', 'Часы', 'Дни', 'Недели', 'Месяцы, кварталы, годы'],
    'Размер патча': [128, 64, 32, 16, 8]
})

print("Рекомендуемые размеры патчей в Moirai:")
print(patch_size_table.to_string(index=False))